In [ ]:
# packages
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import GradientBoostingRegressor as GBR
from sklearn.model_selection import GridSearchCV

# set seed
seed = 5151

### You'll need the California housing data (CAhousing.csv) to complete this exercise. Your goal is to predict the median house value.

In [ ]:
CAhousing = pd.read_csv('data/CAhousing.csv')
CAhousing.head()

In [ ]:
CAhousing.dtypes

In [ ]:
CAhousing.shape

### There is a single categorical variable in this dataset that we will convert to a series of binary variables.

In [ ]:
CAhousing['ocean_proximity'].value_counts()

In [ ]:
# create binary variables for levels which have enough records, using '<1H OCEAN' as base.
CAhousing['ocean_prox_inland'] = (CAhousing.ocean_proximity == 'INLAND').astype(int)
CAhousing['ocean_prox_nearocean'] = (CAhousing.ocean_proximity == 'NEAR OCEAN').astype(int)
CAhousing['ocean_prox_nearbay'] = (CAhousing.ocean_proximity == 'NEAR BAY').astype(int)

# check for intended outcome
print(CAhousing['ocean_prox_inland'].value_counts())
print(CAhousing['ocean_prox_nearocean'].value_counts())
print(CAhousing['ocean_prox_nearbay'].value_counts())

In [ ]:
indep_vars = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
              'total_bedrooms', 'population', 'households', 'median_income', 
              'ocean_prox_inland', 'ocean_prox_nearocean', 'ocean_prox_nearbay']

X = CAhousing[indep_vars]
y = CAhousing['median_house_value']

In [ ]:
X_train, X_test, y_train, y_test, Train, Test = train_test_split(X, y, CAhousing, 
                                                                random_state = seed, 
                                                                test_size = 0.33, 
                                                                shuffle = True)

In [ ]:
X_train.corr().style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
Train.describe()

### We see that `total_bedrooms` has a few missing values, and it is also very correlated with `total_rooms`. Because of this, it's easiest to remove it from our training set.

In [ ]:
indep_vars.remove("total_bedrooms")

### We'll define some helper functions here:

In [ ]:
def mse(y, y_hat):
    resid = y - y_hat
    sq_resid = resid**2
    SSR = sum(sq_resid)
    MSE = SSR / y.shape[0]
    return MSE

def gbm_progress(model_name, X_test, y_test):
    test_error = np.zeros_like(model_name.train_score_)
    for idx, y_ in enumerate(model_name.staged_predict(X_test)):
        test_error[idx] = np.mean((y_test - y_)**2)

    plot_idx = np.arange(model_name.train_score_.shape[0])
    ax = subplots(figsize=(8,8))[1]
    ax.plot(plot_idx, model_name.train_score_, 'b', label='Training')
    ax.plot(plot_idx, test_error, 'r', label='Test')
    ax.legend();

In [ ]:
gbm_basic = GBR(learning_rate=0.1,
               n_estimators=250,
               max_depth=10,
               min_samples_split=2,
               subsample=1,
               random_state=seed)

gbm_basic.fit(X_train[indep_vars], y_train)

In [ ]:
y_hat_basic = gbm_basic.predict(X_test[indep_vars])
mse_basic = mse(y_test, y_hat_basic)
print('test mse: ',mse_basic)

In [ ]:
gbm_progress(gbm_basic, X_test[indep_vars], y_test)

In [ ]:
# Define model
gbm_cv = GBR(random_state=seed)

param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "min_samples_leaf": [1, 5],
    "subsample": [0.8, 1.0]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=gbm_cv,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=3
)

grid_search.fit(X_train[indep_vars], y_train)

In [ ]:
best_model = grid_search.best_estimator_
print(grid_search.best_params_)

In [ ]:
y_pred = best_model.predict(X_test[indep_vars])
mse_better = mse(y_test, y_pred)
print("Test MSE:", mse_better)